# ML-06 Contract Authoring

ML-05 approved input을 상속해 ML-06 `fb_input_feature_contract.csv`를 생성한다. 이 노트북은 parquet feature build를 실행하지 않는다.


## 00 Path Setup


In [6]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'Graph_AML' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURE_BUILD_DIR = PROJECT_ROOT / 'ml' / 'ml-06' / '00_feature_build'
if str(FEATURE_BUILD_DIR) not in sys.path:
    sys.path.insert(0, str(FEATURE_BUILD_DIR))

PROJECT_ROOT, FEATURE_BUILD_DIR


(PosixPath('/mnt/d/AML_git/Graph_AML'),
 PosixPath('/mnt/d/AML_git/Graph_AML/ml/ml-06/00_feature_build'))

## 01 Configuration


In [7]:
from ml_06_fb_catalog import build_input_contract, ratio_base_columns_from_contract, selected_feature_columns, validate_input_contract
from ml_06_fb_io import (
    DEFAULT_FEATURE_COLUMNS_PATH,
    DEFAULT_INPUT_CATEGORY_VALUES_PATH,
    DEFAULT_INPUT_CONTRACT_PATH,
    DEFAULT_INPUT_PATH,
    DEFAULT_SOURCE_CONTRACT_PATH,
    DEFAULT_SOURCE_ENCODING_MANIFEST_PATH,
    load_feature_columns,
    read_json,
    save_dataframe_csv,
    write_json,
)

ARTIFACT_PREFIX = 'ml_06__r00'
PARENT_ARTIFACT_PREFIX = 'ml_05__r00'
APPROVE_VARIANT = 'clip_p9999'
OVERWRITE_CONTRACT_OUTPUTS = False

print('DEFAULT_INPUT_PATH              :', DEFAULT_INPUT_PATH)
print('DEFAULT_SOURCE_CONTRACT_PATH    :', DEFAULT_SOURCE_CONTRACT_PATH)
print('DEFAULT_FEATURE_COLUMNS_PATH    :', DEFAULT_FEATURE_COLUMNS_PATH)
print('DEFAULT_INPUT_CONTRACT_PATH     :', DEFAULT_INPUT_CONTRACT_PATH)
print('DEFAULT_INPUT_CATEGORY_VALUES   :', DEFAULT_INPUT_CATEGORY_VALUES_PATH)
print('APPROVE_VARIANT                 :', APPROVE_VARIANT)
print('OVERWRITE_CONTRACT_OUTPUTS      :', OVERWRITE_CONTRACT_OUTPUTS)


DEFAULT_INPUT_PATH              : /mnt/d/AML_git/Graph_AML/ml/ml-05/ml_inputs/r00/ml_05__r00_Xy_all.parquet
DEFAULT_SOURCE_CONTRACT_PATH    : /mnt/d/AML_git/Graph_AML/ml/ml-05/ml_inputs/r00/ml_05__r00_fb_output_feature_contract_approve.csv
DEFAULT_FEATURE_COLUMNS_PATH    : /mnt/d/AML_git/Graph_AML/ml/ml-05/ml_outputs/r00/ml_05__r00__d00-optuna_t25_feature_columns.json
DEFAULT_INPUT_CONTRACT_PATH     : /mnt/d/AML_git/Graph_AML/ml/ml-06/fb_inputs/r00/ml_06__r00_fb_input_feature_contract.csv
DEFAULT_INPUT_CATEGORY_VALUES   : /mnt/d/AML_git/Graph_AML/ml/ml-06/fb_inputs/r00/ml_06__r00_category_values.json
APPROVE_VARIANT                 : clip_p9999
OVERWRITE_CONTRACT_OUTPUTS      : False


## 02 Source Schema and Parent Contract


In [8]:
source_schema = pq.read_schema(DEFAULT_INPUT_PATH)
source_columns = list(source_schema.names)
source_contract = pd.read_csv(DEFAULT_SOURCE_CONTRACT_PATH, encoding='utf-8-sig', dtype={'used_in_ml': 'string'})
source_feature_columns = load_feature_columns(DEFAULT_FEATURE_COLUMNS_PATH)

print('source_column_count        :', len(source_columns))
print('source_contract_rows      :', len(source_contract))
print('source_feature_count      :', len(source_feature_columns))
display(pd.DataFrame({'column_name': source_columns, 'arrow_type': [str(source_schema.field(c).type) for c in source_columns]}).head(20))


source_column_count        : 275
source_contract_rows      : 275
source_feature_count      : 236


,column_name,arrow_type
0,tx_id,string
1,timestamp,timestamp[us]
2,split,string
3,label,int8
4,sender_account,string
5,receiver_account,string
6,Timestamp,string
7,From Bank,string
8,Account,string
9,To Bank,string


## 03 Build ML-06 Input Contract


In [9]:
contract_df = build_input_contract(
    source_contract,
    feature_columns=source_feature_columns,
    available_columns=source_columns,
    artifact_prefix=ARTIFACT_PREFIX,
    parent_artifact_prefix=PARENT_ARTIFACT_PREFIX,
    approve_variant=APPROVE_VARIANT,
)

validation = validate_input_contract(contract_df, artifact_prefix=ARTIFACT_PREFIX, available_columns=source_columns)
ratio_bases = ratio_base_columns_from_contract(contract_df, source_columns)

print('contract_rows          :', validation.total_rows)
print('selected_count        :', validation.selected_count)
print('ratio_base_count      :', len(ratio_bases))
print('selected_clip_count   :', sum('__clip_train_p9999' in c for c in validation.selected_columns))
print('selected_base_ratios  :', sum(c in ratio_bases for c in validation.selected_columns))

display(contract_df.loc[contract_df['ml_06_policy'].astype(str).str.contains('ratio', na=False), [
    'column_name', 'used_in_ml', 'source_column', 'ml_06_ratio_base_column', 'ml_06_ratio_variant', 'feature_group'
]].head(40))


contract_rows          : 317
selected_count        : 236
ratio_base_count      : 21
selected_clip_count   : 21
selected_base_ratios  : 0


,column_name,used_in_ml,source_column,ml_06_ratio_base_column,ml_06_ratio_variant,feature_group
29,amount__paid_recv_ratio,FALSE,amount__paid_recv_ratio,amount__paid_recv_ratio,original,parent_ml04_selected
90,timehist__sender__out__amount__cur_vs_mean_rat...,FALSE,timehist__sender__out__amount__cur_vs_mean_rat...,timehist__sender__out__amount__cur_vs_mean_rat...,original,parent_ml04_selected
91,timehist__sender__out__amount__cur_vs_mean_rat...,FALSE,timehist__sender__out__amount__cur_vs_mean_rat...,timehist__sender__out__amount__cur_vs_mean_rat...,original,parent_ml04_selected
92,timehist__sender__out__amount__cur_vs_mean_rat...,FALSE,timehist__sender__out__amount__cur_vs_mean_rat...,timehist__sender__out__amount__cur_vs_mean_rat...,original,parent_ml04_selected
93,timehist__receiver__in__amount__cur_vs_mean_ra...,FALSE,timehist__receiver__in__amount__cur_vs_mean_ra...,timehist__receiver__in__amount__cur_vs_mean_ra...,original,parent_ml04_selected
94,timehist__receiver__in__amount__cur_vs_mean_ra...,FALSE,timehist__receiver__in__amount__cur_vs_mean_ra...,timehist__receiver__in__amount__cur_vs_mean_ra...,original,parent_ml04_selected
95,timehist__receiver__in__amount__cur_vs_mean_ra...,FALSE,timehist__receiver__in__amount__cur_vs_mean_ra...,timehist__receiver__in__amount__cur_vs_mean_ra...,original,parent_ml04_selected
207,pair__sender_receiver__bidirectional__amount_r...,FALSE,pair__sender_receiver__bidirectional__amount_r...,pair__sender_receiver__bidirectional__amount_r...,original,parent_ml04_selected
214,pair__sender_receiver__bidirectional__amount_r...,FALSE,pair__sender_receiver__bidirectional__amount_r...,pair__sender_receiver__bidirectional__amount_r...,original,parent_ml04_selected
221,pair__sender_receiver__bidirectional__amount_r...,FALSE,pair__sender_receiver__bidirectional__amount_r...,pair__sender_receiver__bidirectional__amount_r...,original,parent_ml04_selected


## 04 Save and Validate


In [10]:
selected = set(selected_feature_columns(contract_df))
source_manifest = read_json(DEFAULT_SOURCE_ENCODING_MANIFEST_PATH)
source_category_values = source_manifest.get('category_values', {}) if isinstance(source_manifest, dict) else {}
category_values = {
    column: values
    for column, values in source_category_values.items()
    if column in selected
}

save_dataframe_csv(contract_df, DEFAULT_INPUT_CONTRACT_PATH, overwrite=OVERWRITE_CONTRACT_OUTPUTS)
write_json(
    DEFAULT_INPUT_CATEGORY_VALUES_PATH,
    {
        'artifact_prefix': ARTIFACT_PREFIX,
        'parent_artifact_prefix': PARENT_ARTIFACT_PREFIX,
        'fit_split': 'inherited_or_empty',
        'source_contract_path': str(DEFAULT_INPUT_CONTRACT_PATH),
        'category_values': category_values,
    },
    overwrite=OVERWRITE_CONTRACT_OUTPUTS,
)

saved_contract = pd.read_csv(DEFAULT_INPUT_CONTRACT_PATH, encoding='utf-8-sig', dtype={'used_in_ml': 'string'})
saved_validation = validate_input_contract(saved_contract, artifact_prefix=ARTIFACT_PREFIX, available_columns=source_columns)

print('contract saved', DEFAULT_INPUT_CONTRACT_PATH)
print('category values saved', DEFAULT_INPUT_CATEGORY_VALUES_PATH)
print('total_rows', saved_validation.total_rows)
print('selected_count', saved_validation.selected_count)
display(saved_contract)


contract saved /mnt/d/AML_git/Graph_AML/ml/ml-06/fb_inputs/r00/ml_06__r00_fb_input_feature_contract.csv
category values saved /mnt/d/AML_git/Graph_AML/ml/ml-06/fb_inputs/r00/ml_06__r00_category_values.json
total_rows 317
selected_count 236


,contract_version,artifact_prefix,column_name,used_in_ml,source_column,column_origin,encoding,build_action,build_in_fb,xgb_feature_type,...,observed_dtype,missing_count,missing_rate,unknown_category_count,fit_split,note,ml_06_policy,ml_06_ratio_base_column,ml_06_ratio_variant,recency_sentinel_policy
0,1,ml_06__r00,tx_id,FALSE,tx_id,preprocessing,passthrough,carry_forward,False,NaN,...,object,NaN,NaN,NaN,NaN,source parquet column preserved; not selected ...,NaN,NaN,NaN,NaN
1,1,ml_06__r00,timestamp,FALSE,timestamp,preprocessing,passthrough,carry_forward,False,NaN,...,datetime64[us],NaN,NaN,NaN,NaN,source parquet column preserved; not selected ...,NaN,NaN,NaN,NaN
2,1,ml_06__r00,split,FALSE,split,preprocessing,passthrough,carry_forward,False,NaN,...,string,NaN,NaN,NaN,NaN,source parquet column preserved; not selected ...,NaN,NaN,NaN,NaN
3,1,ml_06__r00,label,FALSE,label,preprocessing,passthrough,carry_forward,False,NaN,...,int8,NaN,NaN,NaN,NaN,source parquet column preserved; not selected ...,NaN,NaN,NaN,NaN
4,1,ml_06__r00,sender_account,FALSE,sender_account,preprocessing,passthrough,carry_forward,False,NaN,...,object,NaN,NaN,NaN,NaN,source parquet column preserved; not selected ...,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
312,ml_06_ratio_transform_v1,ml_06__r00,passflow__sender__in_then_out__current_to_last...,TRUE,passflow__sender__in_then_out__current_to_last...,ml_06_feature_build,passthrough,build,True,q,...,float32,NaN,NaN,NaN,NaN,pure ratio transform; train-only clipping for ...,ratio_transform,passflow__sender__in_then_out__current_to_last...,clip_p9999,NaN
313,ml_06_ratio_transform_v1,ml_06__r00,passflow__sender__in_then_out__current_to_last...,FALSE,passflow__sender__in_then_out__current_to_last...,ml_06_feature_build,passthrough,build,True,q,...,float32,NaN,NaN,NaN,NaN,pure ratio transform; train-only clipping for ...,ratio_transform,passflow__sender__in_then_out__current_to_last...,log1p,NaN
314,ml_06_ratio_transform_v1,ml_06__r00,passflow__sender__in_then_out__current_to_last...,TRUE,passflow__sender__in_then_out__current_to_last...,ml_06_feature_build,passthrough,build,True,q,...,float32,NaN,NaN,NaN,NaN,pure ratio transform; train-only clipping for ...,ratio_transform,passflow__sender__in_then_out__current_to_last...,clip_p9999,NaN
315,ml_06_ratio_transform_v1,ml_06__r00,passflow__sender__in_then_out__current_to_last...,FALSE,passflow__sender__in_then_out__current_to_last...,ml_06_feature_build,passthrough,build,True,q,...,float32,NaN,NaN,NaN,NaN,pure ratio transform; train-only clipping for ...,ratio_transform,passflow__sender__in_then_out__current_to_last...,log1p,NaN
